In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import netCDF4 as nc4
from numpy.polynomial.polynomial import polyroots
from scipy.integrate import solve_ivp
from scipy.special import gamma, loggamma, gammainc, gammaincc
import matplotlib.animation as animation

# Definitions of constants, grids, and utilities

## Physical constants

In [ ]:
# Density of water in kg/m^3
rhow = 997.
# Conversion factor between diameter^3 and drop mass, kg/m^3
d3_to_m = np.pi * rhow / 6.

## Vertical discretization

In [ ]:
# Top of rain shaft (m)
z_top = 2000.
# Number of vertical levels
nz = 200
# Level thickness (m)
dz = z_top / nz
# Level heights for plotting (m)
zs = np.flipud(np.linspace(0.5*dz, z_top - 0.5*dz, nz))

## Bin model size discretization

In [ ]:
# Maximum rain drop size, in meters.
d_max = 5000.e-6
# Number of discrete sizes in bin model.
nbin = 1000
# "Boundaries" of size bins (not actually used by model).
bound_ds = np.linspace(0., d_max, nbin+1)
# Uniform width of size bins.
dd = bound_ds[1] - bound_ds[0]
# Midpoints of size bins.
ds = 0.5 * (bound_ds[1:] + bound_ds[:-1])

## Solution evaluation times (requested from SciPy solver)

In [ ]:
# Simulation length (seconds)
tmax = 3600.
# Number of time steps in evaluation
nt = 240
# Times at which to return the approximate solution (seconds)
t_eval = np.linspace(0., tmax, nt+1)
# Time between evaluations.
dt_eval = t_eval[1]

In [ ]:
# This array is used for some fast calculations/plotting.
ds_2d = np.repeat(np.reshape(ds, (1, nbin)), nz, axis=0)
ds_3d = np.repeat(np.reshape(ds_2d, (nz, nbin, 1)), nt+1, axis=2)

## Velocity formula

In [ ]:
def p3_v(d):
    """Return velocity in m/s given diameter in meters using P3 formula.

    Dependence of velocity on air density is neglected.
    """
    m_cgs = 1000. * d3_to_m * d**3
    if d <= 134.43e-6:
        return 4579.5 * m_cgs**(2./3.)
    elif d <= 1511.64e-6:
        return 49.62 * m_cgs**(1./3.)
    elif d <= 3477.84e-6:
        return 17.32 * m_cgs**(1./6.)
    else:
        return 9.17

To get an idea for the velocity formula, let's graph over the given grid.

In [ ]:
vs = np.array([p3_v(d) for d in ds])
plt.plot(1.e3*ds, vs)
plt.xlabel("Diameter (mm)")
plt.ylabel("Terminal velocity (m/s)")
plt.savefig('d_vs_v.png')

## Formulas for exponential distributions

In [ ]:
def m0_m3_to_lambda(m0, m3):
    """Return exponential distribution parameters given M0 and M3.

    Arguments:
    m0 - The 0th moment in #/kg or #/m^3.
    m3 - The 3rd moment in m^3/kg or m^3/m^3 (must be consistent with m0).

    Returns size parameter (lambda) for an exponential distribution.

    If either m0 or m3 are equal to zero, returns 1 as an arbitrary value.
    This avoids floating point exceptions, but may hide coding errors or
    numerical problems (and is admittedly a lazy solution for this demo).
    """
    if m0 == 0. or m3 == 0:
        return 1.
    return (6. * m0 / m3)**(1./3.)

In [ ]:
def exponential_distribution(d, m0, lam):
    """Exponential distribution PDF over D, given parameters m0 and lam.

    Arguments:
    d - Diameter in meters.
    m0 - The 0th moment in #/kg or #/m^3.
    lam - The exponential distribution size parameter (lambda).

    Returns the PDF value at d, in #/kg/m or #/m^3 (matching the input m0).
    """
    return m0 * lam * np.exp(-lam * d)

In [ ]:
def factorial(n):
    """Simple recursive formula for the factorial of n."""
    if n == 0:
        return 1
    return n * factorial(n-1)

In [ ]:
def exponential_distribution_moment(i, m0, lam):
    """Calculate the i-th moment of an exponential distribution.

    Arguments:
    i - Moment to return.
    m0 - The 0th moment in #/kg or #/m^3.
    lam - The exponential distribution size parameter (lambda).

    The returned moment has units that depend on the input M0 and lambda
    parameters, i.e. units of M0/lambda**i.
    """
    return m0 * factorial(i) / lam**i

## Formulas for Gamma distributions

In [ ]:
def m0_m3_m6_to_lambda_nu(m0, m3, m6):
    """Return gamma distribution parameters given M0, M3, and M6.

    Arguments:
    m0 - The 0th moment in #/kg or #/m^3.
    m3 - The 3rd moment in m^3/kg or m^3/m^3.
    m6 - The 6th moment in m^6/kg or m^6/m^3.

    Units for all moments must be consistent with each other.

    Returns parameters (lambda, nu) for a gamma distribution.

    If any input moments are zero, then (1., 1.) is returned.
    This avoids floating point exceptions, but may hide coding errors or
    numerical problems (and is admittedly a lazy solution for this demo).

    Additionally, the value of the width parameter k=m0*m6/m3**2 is capped at
    1.01, which effectively limits nu to no more than 902. This is not a major
    concern, because:
        (a) larger values of nu would produce a very similar (very narrow)
            distribution to nu=902 anyway, so calculated values such as fall
            speed should not be affected very much by this limiter,
        (b) larger values of nu are unrealistic anyway, at least over volumes
            large enough for bulk and bin models to be statistically valid, and
        (c) larger values of nu would tend to cause overflows in later
            computations, as nu is generally used as an exponent and a gamma
            function input.

    It is also common to use a limiter to enforce nu >= 1 (k >= 20), which
    is intended to prevent unrealistically broad distributions. No such limiter
    is implemented here, because this demonstration does not contain any
    microphysical processes that would broaden the distribution.
    """
    if m0 == 0. or m3 == 0. or m6 == 0.:
        return 1., 1.
    k = max(m0 * m6 / (m3 * m3), 1.01)
    coefs = np.array([-60., 2*k - 47., 3*k - 12., k - 1.])
    roots = polyroots(coefs)
    # Find the positive real root. It appears that there will only ever be one
    # root with positive real part; check for near-zero imaginary part anyway.
    nu = -1.
    for root in roots:
        if np.real(root) > 0. and np.abs(np.imag(root)) < 1.e-10:
            nu = np.real(root)
    assert nu > 0., f"solving for nu failed with moments={m0},{m3},{m6}"
    lam = (m0 / m3 * nu * (nu+1.) * (nu+2.))**(1./3.)
    return lam, nu

In [ ]:
def gamma_distribution(d, m0, lam, nu):
    """Gamma distribution PDF over D, given parameters m0, lambda, and nu.

    Arguments:
    d - Diameter in meters.
    m0 - The 0th moment in #/kg or #/m^3.
    lam - The gamma distribution size parameter (lambda).
    nu - The gamma distribution shape parameter.

    Returns the PDF value at d, in #/kg/m or #/m^3 (matching the input m0).
    """
    # For dealing with large nu values, calculate log of the distribution
    # first, then exponentiate it.
    log_dist = -lam * d + nu * np.log(lam) + (nu-1.) * np.log(d) - loggamma(nu)
    return m0 * np.exp(log_dist)

In [ ]:
# Precalculate some constants, for efficiency.
m_fac = 1000. * np.pi * rhow/6.
term1_coef = 4579.5 * m_fac**(2./3.)
term2_coef = 49.62 * m_fac**(1./3.)
term3_coef = 17.32 * m_fac**(1./6.)

def v_moment_gamma(i, lam, nu):
    """Find moment-weighted fall speeds for a gamma distribution.

    Arguments:
    i - The moment to calculate a mean fall speed for.
    lam - The gamma distribution size parameter (lambda).
    nu - The gamma distribution shape parameter.

    Returns the i-th moment-weighted fall speed in m/s using the P3
    parameterization of terminal velocity.
    """
    term1 = term1_coef / lam**2 * gammainc(nu+i+2, 134.43e-6*lam)
    # Add correction factor for the fact that gammainc is the regularized function
    # (thus off by a factor of gamma(nu+i+2)), and divide by gamma(nu+i).
    term1 *= (nu+i+1) * (nu+i)
    term2 = term2_coef / lam \
        * (gammaincc(nu+i+1, 134.43e-6*lam) - gammaincc(nu+i+1, 1511.64e-6*lam))
    # Gamma correction factors as in term1.
    term2 *= nu+i
    term3 = term3_coef / np.sqrt(lam) \
        * (gammaincc(nu+i+0.5, 1511.64e-6*lam) - gammaincc(nu+i+0.5, 3477.84e-6*lam))
    # Gamma correction factors as in other terms.
    # This formulation reduces chance of overflow.
    term3 *= np.exp(loggamma(nu+i+0.5)-loggamma(nu+i))
    term4 = 9.17 * gammaincc(nu+i, 3477.84e-6*lam)
    # No gamma correction factor necessary for term4.
    return term1 + term2 + term3 + term4

## Moment-weighted velocities for bulk models

In [ ]:
def vs_2m(ms):
    """Calculate 0th and 3rd moment-weighted fall speeds.

    Arguments:
    ms - Iterable containing M0 and M3, in that order. Units of M3/M0 must be
        cubic meters.

    Returns a size-2 numpy vector containing the mean fall speeds for M0 and
    M3, in that order.
    """
    m0 = ms[0]
    m3 = ms[1]
    if m3 == 0. or m0 == 0.:
        return np.zeros((2,))
    lam = m0_m3_to_lambda(m0, m3)
    v0 = v_moment_gamma(0, lam, 1)
    v3 = v_moment_gamma(3, lam, 1)
    return np.array((v0, v3))

Plot moment-weighted fall speeds for exponential distribution with a range of lambda values.

In [ ]:
plot_v_2ms = np.zeros((nbin,2))
for i in range(nbin):
    plot_v_2ms[i,:] = vs_2m([1., ds[i]**3 * 6])
plt.plot(1000.*ds, plot_v_2ms[:,0], 'b', label='$v_0$')
plt.plot(1000.*ds, plot_v_2ms[:,1], 'grey', label='$v_3$')
plt.xlabel('$\overline{D}$ (mm)')
plt.ylabel('Fall speed (m/s)')
plt.legend()
plt.savefig("fall_speed_2m.png")

In [ ]:
def vs_3m(ms):
    """Calculate 0th, 3rd, and 6th moment-weighted fall speeds.

    Arguments:
    ms - Iterable containing M0, M3, and M6 in that order. Units of M3/M0 and
        of M6/M3 must be the same, and must be cubic meters.

    Returns a size-3 numpy vector containing the mean fall speeds for M0, M3,
    and M6, in that order.
    """
    m0 = ms[0]
    m3 = ms[1]
    m6 = ms[2]
    if m0 == 0. or m3 == 0. or m6 == 0.:
        return np.zeros((3,))
    lam, nu = m0_m3_m6_to_lambda_nu(m0, m3, m6)
    v0 = v_moment_gamma(0, lam, nu)
    v3 = v_moment_gamma(3, lam, nu)
    v6 = v_moment_gamma(6, lam, nu)
    return np.array((v0, v3, v6))

Plot moment-weighted fall speeds for gamma distribution with a range of lambda values,
and nu=1 or nu=3.

In [ ]:
plot_v_3ms = np.zeros((nbin,3,2))
for i in range(nbin):
    plot_v_3ms[i,:,0] = vs_3m([1., ds[i]**3 * 6, ds[i]**6 * 720])
    plot_v_3ms[i,:,1] = vs_3m([1., (ds[i]/3)**3 * 60, (ds[i]/3)**6 * 20160])
plt.plot(1000.*ds, plot_v_3ms[:,0,0], 'b', label=r"$v_0$ ($\nu=1$)")
plt.plot(1000.*ds, plot_v_3ms[:,1,0], 'grey', label=r"$v_3$ ($\nu=1$)")
plt.plot(1000.*ds, plot_v_3ms[:,2,0], 'orange', label=r"$v_6$ ($\nu=1$)")
plt.plot(1000.*ds, plot_v_3ms[:,0,1], 'b', linestyle='--', label=r"$v_0$ ($\nu=3$)")
plt.plot(1000.*ds, plot_v_3ms[:,1,1], 'grey', linestyle='--', label=r"$v_3$ ($\nu=3$)")
plt.plot(1000.*ds, plot_v_3ms[:,2,1], 'orange', linestyle='--', label=r"$v_6$ ($\nu=3$)")
plt.xlabel(r'$\overline{D}$ (mm)')
plt.ylabel('Fall speed (m/s)')
plt.legend()
plt.savefig("fall_speed_3m.png")

## Plotting utility

In [ ]:
def plot_snapshots(time_indices, quantities, xlimits, time_labels,
                   quantity_labels, model_labels, y_label, colors,
                   legend_ax):
    """Produce 2D plot of quantity snapshots.

    Arguments:
    time_indices - Indexes for time snapshots at which to make plots.
    quantities - Collection of collections of arrays containing data to plot.
        Outermost index should vary over quantities to plot, next index should
        be over models. Each array so indexed should be an nz by nt+1 array.
    xlimits - List of tuples containing lower and upper bounds for quantity
        ranges. Can specify `None` instead of a tuple to use default range.
    time_labels - Text to label snapshots (e.g. 't=4 min').
    quantity_labels - Names of quantities to label x-axis.
    model_labels - Names of models for legend.
    y_label - Label for y-axis.
    colors - Colors to use for different models.
    legend_ax - A tuple used to specify the location of the legend. E.g. (2, 3)
        will select the axis 3rd from the top, 4th from the left.

    Returns a tuple `(fig, axs)` containing the figure and axes produced.
    """
    # Number of time snapshots.
    plot_nt = len(time_indices)
    assert 0 <= legend_ax[0] < plot_nt
    # Number of quantities.
    nq = len(quantities)
    assert 0 <= legend_ax[1] < nq
    # Number of models.
    nm = len(model_labels)
    for q in range(nq):
        assert len(quantities[q]) == nm
    # Set up figure and axes.
    fig, axs = plt.subplots(plot_nt, nq, sharex='col', sharey=True,
                            figsize=(1.5*plot_nt, 1. + 1.5*nq))
    # Plot line width.
    lw = 1.
    # Place time label text on plots in center column.
    for i in range(plot_nt):
        axs[i,nq//2].text(0.08, 0.85, time_labels[i],
                          transform=axs[i,nq//2].transAxes)
    # Set ranges for quantities where the default is not ideal.
    for q, xlimit in enumerate(xlimits):
        if xlimit is not None:
            axs[0,q].set_xlim(xlimit)
    # Loop over snapshot times.
    for i in range(plot_nt):
        ti = time_indices[i]
        # Loop over models.
        for m in range(3):
            # Loop over quantities.
            for q in range(5):
                # Choose color based on model, cycle through colors if
                # there are not enough.
                axs[i,q].plot(quantities[q][m][:,ti], zs,
                              '-', lw=lw, color=colors[m % len(colors)],
                              label=model_labels[m])
    # Legend and axes labels.
    axs[legend_ax[0], legend_ax[1]].legend()
    for i in range(plot_nt):
        axs[i,0].set_ylabel(y_label)
    for q, q_label in enumerate(quantity_labels):
        axs[-1, q].set_xlabel(q_label)
    return fig, axs

# Upper boundary conditions

## For the exponential upper boundary condition

In [ ]:
# Rain number mixing ratio used for initial/boundary conditions (#/kg)
ibc_nr = 1000.
# Rain mass mixing ratio used for initial/boundary conditions (kg/kg)
ibc_qr = 1.e-4
# Density used to convert between number/mass concentrations and moments (kg/m^3)
convert_rho = 1.
# Convert nr to M0 per volume (#/m^3)
ibc_m0 = convert_rho * ibc_nr
# Convert qr to M3 per volume (m^3/m^3)
ibc_m3 = convert_rho * ibc_qr / d3_to_m

In [ ]:
# The upper M0 and M3 values are sufficient for the 2-moment model,
# but the 3-moment model needs an upper boundary condition for M6...
ibc_lam = m0_m3_to_lambda(ibc_m0, ibc_m3)
ibc_m6 = exponential_distribution_moment(6, ibc_m0, ibc_lam)
# ...and the bin model needs a boundary condition for each diameter.
ibc_dists = exponential_distribution(ds, ibc_m0, ibc_lam)

Plot the drop size distribution used as the upper boundary condition.

In [ ]:
plt.semilogy(ds, ibc_dists)
plt.xlabel("Diameter (m)")
plt.ylabel("Number density (#$/m^4$)")
plt.savefig("ibc_exponential.png")

## For the gamma upper boundary condition

In [ ]:
# Calculate what M6 would be for nu=3 and the same M0 and M3 as used above.
ibc_gam_m6 = ibc_m3**2 / ibc_m0 * (6*7*8)/(3*4*5)
# Reverse engineer the gamma distribution scale parameter.
ibc_gam_lam = (60. * ibc_m0 / ibc_m3)**(1./3.)
ibc_gam_dists = gamma_distribution(ds, ibc_m0, ibc_gam_lam, 3.)

Plot the bin model upper boundary condition for the two cases.

In [ ]:
plt.plot(ds, ibc_dists, 'k', label='Exponential')
plt.plot(ds, ibc_gam_dists, 'k--', label='Gamma')
plt.xlabel("Diameter (m)")
plt.ylabel("Number density (#$/m^4$)")
plt.legend()
plt.savefig("ibcs.png")

# Solution calculations

## Bin model solve

In [ ]:
# Initial condition for bin model.
y0 = np.zeros((nbin*nz,))

In [ ]:
def f_bin(_, y, ibc_dists):
    """Calculate the rate of change of number density using the bin model.

    Arguments:
    _ - Time input (ignored)
    y - Vector of number densities of size nbin*nz. This can be viewed as a
        flattened version of an array of shape `(nz, nbin)`.
    ibc_dists - Vector of size nbin, providing the number density at the upper
        boundary for each diameter.

    Returns a vector of rates of change, with same size and layout as y.
    """
    dy = np.zeros(y.shape)
    # Apply incoming mass flux from upper boundary condition.
    flux_top = ibc_dists * vs
    dy[:nbin] = flux_top / dz
    # Loop over internal interfaces, subtracting each flux from layer above and
    # adding it to the layer below.
    for k in range(nz-1):
        flux = y[k*nbin:(k+1)*nbin] * vs
        dy[k*nbin:(k+1)*nbin] -= flux / dz
        dy[(k+1)*nbin:(k+2)*nbin] += flux / dz
    # Remove flux of mass out the bottom of the model domain.
    flux_bot = y[(nz-1)*nbin:] * vs
    dy[(nz-1)*nbin:] -= flux_bot / dz
    return dy

In [ ]:
# Run solver for exponential upper boundary condition.
sol = solve_ivp(f_bin, (0., 3600.), y0, t_eval=t_eval, args=(ibc_dists,))
# Reshape the solution output into a more useful form.
bin_evolution = np.reshape(sol.y, (nz, nbin, nt+1))

In [ ]:
# Run solver for gamma upper boundary condition.
sol_gam = solve_ivp(f_bin, (0., 3600.), y0, t_eval=t_eval, atol=1.e-11, args=(ibc_gam_dists,))
# Reshape output into a more useful form.
bin_evolution_gam = np.reshape(sol_gam.y, (nz, nbin, nt+1))

## 2-moment model solve

In [ ]:
# Initial condition for 2-moment model.
y0_2m = np.zeros((2*nz,))

In [ ]:
def f_2m(_, y, ibc_m0, ibc_m3):
    """Calculate the rate of change of moments using the 2-moment bulk model.

    Arguments:
    _ - Time input (ignored)
    y - Vector of moments of size 2*nz. This can be viewed as a
        flattened version of an array of shape `(nz, 2)`.
    ibc_m0 - Scalar value of M0 used as upper boundary condition.
    ibc_m3 - Scalar value of M3 used as upper boundary condition.

    Returns a vector of rates of change, with same size and layout as y.

    To avoid floating point exceptions, negative values in y are treated as if
    they were zero.
    """
    # Note limiter to avoid ending up with NaN when input moments are negative.
    y_limited = np.maximum(y, 0.)
    dy = np.zeros(y.shape)
    # Apply incoming mass flux from upper boundary condition.
    ibc_ms = np.array((ibc_m0, ibc_m3))
    vs = vs_2m(ibc_ms)
    flux_top = ibc_ms * vs
    dy[:2] = flux_top / dz
    # Loop over internal interfaces, subtracting each flux from layer above and
    # adding it to the layer below.
    for k in range(nz-1):
        vs = vs_2m(y_limited[k*2:(k+1)*2])
        flux = y[k*2:(k+1)*2] * vs
        dy[k*2:(k+1)*2] -= flux / dz
        dy[(k+1)*2:(k+2)*2] += flux / dz
    # Remove flux of mass out the bottom of the model domain.
    vs = vs_2m(y_limited[(nz-1)*2:])
    flux_bot = y[(nz-1)*2:] * vs
    dy[(nz-1)*2:] -= flux_bot / dz
    return dy

In [ ]:
# Define absolute tolerances for solver.
atol = np.repeat(np.array([1.e-4, 1.e-14]), nz)

In [ ]:
# Run solver; for the 2-moment bulk model, the exponential and gamma solutions
# are the same.
sol_2m = solve_ivp(f_2m, (0., 3600.), y0_2m, t_eval=t_eval, atol=atol,
                   args=(ibc_m0, ibc_m3))
# Reshape output into a more useful form.
b2m_evolution = np.reshape(sol_2m.y, (nz, 2, nt+1))

## 3-moment model solve

In [ ]:
# Initial condition for 3-moment model.
y0_3m = np.zeros((3*nz,))

In [ ]:
def f_3m(_, y, ibc_m0, ibc_m3, ibc_m6):
    """Calculate the rate of change of moments using the 3-moment bulk model.

    Arguments:
    _ - Time input (ignored)
    y - Vector of moments of size 3*nz. This can be viewed as a
        flattened version of an array of shape `(nz, 3)`.
    ibc_m0 - Scalar value of M0 used as upper boundary condition.
    ibc_m3 - Scalar value of M3 used as upper boundary condition.
    ibc_m6 - Scalar value of M6 used as upper boundary condition.

    Returns a vector of rates of change, with same size and layout as y.

    To avoid floating point exceptions, negative values in y are treated as if
    they were zero.
    """
    # Note limiter to avoid ending up with NaN when input moments are negative.
    y_limited = np.maximum(y, 0.)
    dy = np.zeros(y.shape)
    # Apply incoming mass flux from upper boundary condition.
    ibc_ms = np.array((ibc_m0, ibc_m3, ibc_m6))
    vs = vs_3m(ibc_ms)
    flux_top = ibc_ms * vs
    dy[:3] = flux_top / dz
    # Loop over internal interfaces, subtracting each flux from layer above and
    # adding it to the layer below.
    for k in range(nz-1):
        vs = vs_3m(y_limited[k*3:(k+1)*3])
        flux = y[k*3:(k+1)*3] * vs
        dy[k*3:(k+1)*3] -= flux / dz
        dy[(k+1)*3:(k+2)*3] += flux / dz
    # Remove flux of mass out the bottom of the model domain.
    vs = vs_3m(y_limited[(nz-1)*3:])
    flux_bot = y[(nz-1)*3:] * vs
    dy[(nz-1)*3:] -= flux_bot / dz
    return dy

In [ ]:
# Define absolute tolerances for solver.
atol = np.repeat(np.array([1.e-4, 1.e-14, 1.e-22]), nz)

In [ ]:
# Run solver for exponential upper boundary condition.
sol_3m = solve_ivp(f_3m, (0., 3600.), y0_3m, t_eval=t_eval, atol=atol,
                   args=(ibc_m0, ibc_m3, ibc_m6))
# Reshape output into a more useful form.
b3m_evolution = np.reshape(sol_3m.y, (nz, 3, nt+1))

In [ ]:
# Run solver for gamma upper boundary condition.
sol_3m_gam = solve_ivp(f_3m, (0., 3600.), y0_3m, t_eval=t_eval, atol=atol,
                       args=(ibc_m0, ibc_m3, ibc_gam_m6))
# Reshape output into a more useful form.
b3m_evolution_gam = np.reshape(sol_3m_gam.y, (nz, 3, nt+1))

# Model comparison plots

## Calculating moments for each model

In [ ]:
# Calculate moments 0, 3, and 6 for the bin model using Riemann sums.
m0_bin = dd*np.sum(bin_evolution, 1)
m3_bin = dd*np.sum(ds_3d**3 * bin_evolution, 1)
m6_bin = dd*np.sum(ds_3d**6 * bin_evolution, 1)
m0_bin_gam = dd*np.sum(bin_evolution_gam, 1)
m3_bin_gam = dd*np.sum(ds_3d**3 * bin_evolution_gam, 1)
m6_bin_gam = dd*np.sum(ds_3d**6 * bin_evolution_gam, 1)

In [ ]:
# 2-moment model has moments 0 and 3 already, calculate 6.
m0_2m = b2m_evolution[:,0,:]
m3_2m = b2m_evolution[:,1,:]
lam = np.zeros((nz,nt+1))
for k in range(nz):
    for it in range(nt+1):
        lam[k,it] = m0_m3_to_lambda(m0_2m[k,it], m3_2m[k,it])
m6_2m = exponential_distribution_moment(6, m0_2m[:,:], lam)

In [ ]:
# 3-moment model has all three moments already.
m0_3m = b3m_evolution[:,0,:]
m3_3m = b3m_evolution[:,1,:]
m6_3m = b3m_evolution[:,2,:]
m0_3m_gam = b3m_evolution_gam[:,0,:]
m3_3m_gam = b3m_evolution_gam[:,1,:]
m6_3m_gam = b3m_evolution_gam[:,2,:]

## Calculating dBZ

In [ ]:
# Utility for decibel conversion.
to_db = lambda x: 10. * np.log10(x)

In [ ]:
# Reference reflectivity is that of a single 1mm drop per m^3, which works out
# to 10^{-18} m^6 / m^3, and thus a shift of 180.
dbz_bin = 180. + to_db(m6_bin)
dbz_2m = 180. + to_db(m6_2m)
dbz_3m = 180. + to_db(m6_3m)
dbz_bin_gam = 180. + to_db(m6_bin_gam)
dbz_3m_gam = 180. + to_db(m6_3m_gam)

## Calculating effective diameter in microns

In [ ]:
effd = lambda m0, m3: 1.e6 * (m3 / m0)**(1./3.)

In [ ]:
effd_bin = effd(m0_bin, m3_bin)
effd_2m = effd(m0_2m, m3_2m)
effd_3m = effd(m0_3m, m3_3m)
effd_bin_gam = effd(m0_bin_gam, m3_bin_gam)
effd_3m_gam = effd(m0_3m_gam, m3_3m_gam)

## Exponential upper boundary condition graphs

Example code for animating evolution of a variable.

In [ ]:
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(autoscale_on=False,
                     xlim=(0., 1.1*ibc_m3),
                     ylim=(0., z_top))
ax.set_xlabel("$M_3$ ($m^3/m^3$)")
ax.set_ylabel("Altitude (m)")
ax.grid()

line_bin, = ax.plot(m3_bin[:,0], zs, 'k-', lw=2)
line_2m, = ax.plot(m3_2m[:,0], zs, 'r-', lw=2)
line_3m, = ax.plot(m3_3m[:,0], zs, 'g-', lw=2)
time_template = 'time = %.1fs'
time_text = ax.text(0.05, 0.9, '', transform=ax.transAxes)
frame_stride = 1
def animate(i):
    ti = frame_stride * i
    line_bin.set_data(m3_bin[:,ti], zs)
    line_2m.set_data(m3_2m[:,ti], zs)
    line_3m.set_data(m3_3m[:,ti], zs)
    time_text.set_text(time_template % t_eval[ti])
    return line_bin, line_2m, line_3m, time_text

num_steps = nt // 2
num_frames = num_steps // frame_stride
ani = animation.FuncAnimation(
    fig, animate, num_frames,
    interval=10000. / num_frames,
    blit=True)
ani.save("m3_evolution_combined.gif")
plt.show()

Produce array of plots.

In [ ]:
# Determine indices of times to plot.
time_indices = [4, 16, 32, 60, 120]
# Quantities to plot.
quantities = [
    [m0_bin, m0_2m, m0_3m],
    [m3_bin, m3_2m, m3_3m],
    [m6_bin, m6_2m, m6_3m],
    [dbz_bin, dbz_2m, dbz_3m],
    [effd_bin, effd_2m, effd_3m],
]
# Ranges for quantities where default x limits are not ideal.
xlimits = [
    None,
    None,
    (0., 1.1*ibc_m6),
    (-20., 80.),
    (0., 8.e6*(ibc_m3/ibc_m0)**(1./3.)),
]
# Text labels to label snapshots.
times_minutes = [ti * dt_eval / 60 for ti in time_indices]
time_labels = [f't={t_min} min' for t_min in times_minutes]
# Quantity labels
quantity_labels = [
    '$M_0$',
    '$M_3$',
    '$M_6$',
    'dBZ',
    '$\sqrt[3]{M_3/M_0} (\mu m)$'
]
# Model labels
model_labels = ['Bin', '2M', '3M']
# y-axis label
y_label = 'Altitude (m)'
# Colors to use for different models.
colors = ['k', 'r', 'g']
# Plot axis on which to place legend.
legend_ax = (4, 4)

In [ ]:
fig, axs = plot_snapshots(time_indices, quantities, xlimits, time_labels,
    quantity_labels, model_labels, y_label, colors,
    legend_ax)
# Add plots 
for i, ti in enumerate(time_indices):
    height = 2000. - ti*dt_eval*9.17
    if height > 0:
        for ax in axs[i,:]:
            ax.hlines([height],
                      ax.get_xlim()[0], ax.get_xlim()[1],
                      'k', '--')
plt.savefig('exponential_stats.png')

Zoom in on 2-moment noise.

In [ ]:
plt.plot(m3_2m[:,1], zs, 'r-', lw=1)
plt.plot(m3_2m[:,2], zs, 'r-', lw=1)
plt.plot(m3_2m[:,3], zs, 'r-', lw=1)
plt.plot(m3_2m[:,4], zs, 'r-', lw=1)
plt.plot(m3_2m[:,5], zs, 'r-', lw=1)
heights = 2000. - sol.t[1:6]*9.17
plt.hlines(heights,
           0, 2.e-7,
           'k', '--')
plt.ylim((1000., 2000.))
plt.xlabel('$M_3$ ($m^3/m^3$)')
plt.ylabel('Altitude (m)')
plt.savefig('noise_2m.png')

## Gamma upper boundary condition graphs

In [ ]:
# Quantities to plot.
quantities_gam = [
    [m0_bin_gam, m0_2m, m0_3m_gam],
    [m3_bin_gam, m3_2m, m3_3m_gam],
    [m6_bin_gam, m6_2m, m6_3m_gam],
    [dbz_bin_gam, dbz_2m, dbz_3m_gam],
    [effd_bin_gam, effd_2m, effd_3m_gam],
]
# Ranges for quantities where default x limits are not ideal.
xlimits_gam = [
    None,
    None,
    (0., 1.1*ibc_gam_m6),
    (-20., 80.),
    (0., 8.e6*(ibc_m3/ibc_m0)**(1./3.)),
]

In [ ]:
fig, axs = plot_snapshots(time_indices, quantities_gam, xlimits_gam, time_labels,
    quantity_labels, model_labels, y_label, colors,
    legend_ax)
# Add plots 
for i, ti in enumerate(time_indices):
    height = 2000. - ti*dt_eval*9.17
    if height > 0:
        for ax in axs[i,:]:
            ax.hlines([height],
                      ax.get_xlim()[0], ax.get_xlim()[1],
                      'k', '--')
plt.savefig('gamma_stats.png')